# 365 Probabilidades - Dia #148
## Qual a probabilidade de você lembrar do sonho amanhã de manhã?

**Tipo:** Curiosidade · Sono e memória
**Data de publicação:** 2026-11-08
**Ferramenta:** Python
**Decisão analisada:** Lembrar do sonho é sorte, ou é algo que dá para mudar?
**Hashtag:** #365Probabilidades #Dia148

---

### 📖 A História

Todo mundo já acordou com um sonho inteiro na cabeça e, no tempo de pegar o celular, ficou
só com uma sensação. Às vezes nem isso. Sobra a certeza de que sonhou e mais nada.

Há quem diga que nunca sonha. Há quem conte o sonho da noite em detalhes no café da manhã.
A pergunta deste dia é simples. Numa manhã qualquer, qual a chance de você acordar
lembrando do que sonhou? E o que faz essa chance subir ou cair?

---

### 📚 O Conceito: sonhar e lembrar são coisas diferentes

Os estudos de laboratório mostram que quase todo mundo sonha. Quando alguém é acordado
durante o sono REM, relata um sonho com conteúdo em cerca de 85% das vezes; fora do REM,
em cerca de 45%. O que varia de pessoa para pessoa não é tanto sonhar, é guardar o sonho
até o momento de contar.

Entre lembrar e esquecer existe um meio-termo que os pesquisadores chamam de "sonho em
branco": a pessoa sabe que sonhou, mas não consegue dizer nada sobre o sonho.

---

### 🧮 O Modelo

Quatro camadas. A primeira, a segunda e a terceira são contas nossas sobre os dados abertos
do estudo. A quarta traz a segunda fonte.

1. **A manhã.** As 2.900 manhãs de 204 pessoas: quantas acordaram com sonho, com sonho em
   branco ou sem nada. Modelo beta-binomial, que separa a chance de cada pessoa.
2. **O que se diz e o que se mede.** Antes do diário, cada pessoa disse num questionário
   com que frequência lembra dos sonhos. Comparamos a resposta com as 15 manhãs gravadas.
3. **Quem lembra mais.** Atitude em relação aos sonhos, mente que divaga, idade e estação
   do ano.
4. **Outra amostra, outro desenho.** Nielsen 2012, 28.888 pessoas: como a frequência de
   lembrar muda com a idade e o sexo.

**Fonte central:**
- Elce, V., Bergamo, D., Bontempi, G., Pedreschi, B., Bellesi, M., Handjaras, G. &
  Bernardi, G. (2025). "The individual determinants of morning dream recall".
  *Communications Psychology* 3, 25, doi 10.1038/s44271-025-00191-z (lido no PDF).
  217 adultos recrutados na Itália, **204 analisados** (18 a 70 anos, idade média 35,1;
  113 mulheres), 15 manhãs de relato gravado por voz e actígrafo no pulso, de março de 2020
  a março de 2024; **2.900 relatos**. Sonho com conteúdo em 0,58 das manhãs (DP 0,24),
  sonho em branco em 0,14 (DP 0,13), nada em 0,28 (DP 0,22); lembrar de ter sonhado, 0,72.
  Lembrar foi previsto pela atitude em relação aos sonhos (t = 4,13), pela mente que
  divaga (t = 3,76) e pelo sono leve longo (t = 5,04); lembrar o conteúdo, pela idade
  (t = −3,51) e pela vulnerabilidade à interferência (t = −3,48). Menos lembrança no
  inverno do que na primavera (g = 0,59).

**Dados próprios:** arquivo aberto dos autores no OSF (projeto 3qwsm, "Somnieve - Dream
Recall Frequency", fulltable_snapshot-05-Nov-2024.mat), convertido para
`dados148/dia-148-elce2025-manhas.csv` (uma linha por manhã).

**Segunda fonte (outra amostra):**
- Nielsen, T. (2012). "Variations in dream recall frequency and dream theme diversity by
  age and sex". *Frontiers in Neurology* 3, 106. Questionário online, **28.888 pessoas**
  de 10 a 79 anos. A frequência de lembrar sobe da adolescência aos 20 a 29 anos e cai nos
  vinte anos seguintes; nos homens a queda começa aos 30 a 39, nas mulheres aos 40 a 49,
  mais abrupta.

**Contexto de laboratório (citado pelos autores):** Siclari et al. 2013, sonho com conteúdo
em cerca de 85% dos despertares do REM e 45% fora do REM.

**Sem fator ×0,80:** a camada central é diário gravado toda manhã, não proporção de pesquisa.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats, optimize, special

SEMENTE = 42
rng = np.random.default_rng(SEMENTE)

plt.rcParams['font.family'] = 'serif'
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3
plt.rcParams['figure.facecolor'] = 'white'
plt.rcParams['axes.facecolor'] = 'white'

VERMELHO, DOURADO, VERDE, CINZA = '#c0392b', '#c8a84b', '#1a5f5a', '#6b6a64'
RODAPE = 'Fonte: Elce et al. (2025), Communications Psychology 3, 25 · dados abertos no OSF · cálculo próprio | #365Probabilidades'

def vir(x, casas=1):
    return f"{x:.{casas}f}".replace(".", ",")

def mil(n):
    return f"{int(round(n)):,}".replace(",", ".")

def jeffreys(k, n):
    return stats.beta(k + 0.5, n - k + 0.5)

print("✅ Bibliotecas carregadas · semente", SEMENTE)

In [ ]:
# --- DADOS ---
# Convertido do arquivo aberto dos autores (OSF 3qwsm, fulltable_snapshot-05-Nov-2024.mat, tabela data_table)
# relato: 'conteudo' (lembrou o sonho), 'branco' (sabe que sonhou, sem conteúdo), 'nada'
m = pd.read_csv('dados148/dia-148-elce2025-manhas.csv', parse_dates=['data'])
m['lembrou'] = (m['relato'] != 'nada').astype(int)
m['conteudo'] = (m['relato'] == 'conteudo').astype(int)

p = m.groupby('participante').agg(n=('lembrou', 'size'), k=('lembrou', 'sum'), c=('conteudo', 'sum'),
                                  idade=('idade', 'first'), atitude=('atitude_sonhos', 'first'),
                                  divaga=('mente_divaga', 'first'), questionario=('questionario_frequencia', 'first'))
p['taxa'] = p['k'] / p['n']

# Escala do questionário (Schredl 2002): 0 nunca · 1 menos de uma vez por mês · 2 uma vez por mês ·
# 3 duas ou três vezes por mês · 4 uma vez por semana · 5 várias vezes por semana · 6 quase toda manhã
ESCALA = {0: 'nunca', 1: 'menos de 1 vez por mês', 2: '1 vez por mês', 3: '2 ou 3 vezes por mês',
          4: '1 vez por semana', 5: 'várias vezes por semana', 6: 'quase toda manhã'}

NIELSEN = {'n': 28888, 'idades': '10 a 79'}
LAB = {'rem': 0.85, 'nrem': 0.45}

print("=" * 70)
print("  DADOS")
print("=" * 70)
print(f"  Elce et al. 2025 · {mil(len(m))} manhãs de {len(p)} pessoas · {vir(p['n'].mean())} manhãs por pessoa")
print(f"  Idade de {int(p['idade'].min())} a {int(p['idade'].max())} anos, média {vir(p['idade'].mean())}")
print(f"  Período: {m['data'].min():%m/%Y} a {m['data'].max():%m/%Y}")
for r in ['conteudo', 'branco', 'nada']:
    print(f"    {r:<9} {mil((m['relato'] == r).sum()):>6} manhãs  {vir((m['relato'] == r).mean() * 100)}%")
print(f"  Nielsen 2012 · {mil(NIELSEN['n'])} pessoas de {NIELSEN['idades']} anos")
print("=" * 70)

In [ ]:
# --- O MODELO ---

# CAMADA 1 · A manhã (beta-binomial por máxima verossimilhança)
k_tot, n_tot = m['lembrou'].sum(), len(m)
ic_bruto = jeffreys(k_tot, n_tot).interval(0.95)

def nll(par):
    a, b = np.exp(par)
    return -np.sum(special.betaln(p['k'] + a, p['n'] - p['k'] + b) - special.betaln(a, b))
res = optimize.minimize(nll, [0.0, 0.0])
A, B = np.exp(res.x)
pessoa = stats.beta(A, B)
media_bb = A / (A + B); rho = 1 / (A + B + 1)

def prob_k_de_n(k, n):
    return np.exp(special.gammaln(n + 1) - special.gammaln(k + 1) - special.gammaln(n - k + 1)
                  + special.betaln(k + A, n - k + B) - special.betaln(A, B))

# intervalo da média por reamostragem de pessoas
ids = p.index.to_numpy(); boot = []
for _ in range(1000):
    s = p.loc[rng.choice(ids, len(ids))]
    f = lambda par: -np.sum(special.betaln(s['k'].to_numpy() + np.exp(par[0]), s['n'].to_numpy() - s['k'].to_numpy() + np.exp(par[1])) - special.betaln(np.exp(par[0]), np.exp(par[1])))
    a_, b_ = np.exp(optimize.minimize(f, res.x).x); boot.append(a_ / (a_ + b_))
ic_bb = np.percentile(boot, [2.5, 97.5])

print("=" * 70)
print("  CAMADA 1 · A MANHÃ")
print("=" * 70)
print(f"  Todas as manhãs juntas: lembrou de ter sonhado em {k_tot} de {n_tot} = {vir(k_tot / n_tot * 100)}% [{vir(ic_bruto[0] * 100)}; {vir(ic_bruto[1] * 100)}]")
print(f"  Beta-binomial: chance média por manhã {vir(media_bb * 100)}% [{vir(ic_bb[0] * 100)}; {vir(ic_bb[1] * 100)}] · Beta({vir(A, 2)}; {vir(B, 2)})")
print(f"  Correlação entre manhãs da mesma pessoa: {vir(rho, 2)}")
print(f"  Pessoas que lembram em mais de 90% das manhãs: {vir(pessoa.sf(0.9) * 100, 0)}% · em menos da metade: {vir(pessoa.cdf(0.5) * 100, 0)}%")
print(f"  Ninguém do estudo passou as 15 manhãs sem lembrar nada. A menor taxa foi {vir(p['taxa'].min() * 100, 0)}%.")
print(f"  Numa semana: nenhuma manhã {vir(prob_k_de_n(0, 7) * 100)}% · todas as sete {vir(prob_k_de_n(7, 7) * 100)}%")

# CAMADA 2 · O que se diz e o que se mede
diz = p.groupby('questionario').agg(pessoas=('taxa', 'size'), lembrou=('taxa', 'mean'))
rho_s = stats.spearmanr(p['questionario'], p['taxa'])
baixo = p['questionario'] <= 2
print("\n" + "=" * 70)
print("  CAMADA 2 · O QUE SE DIZ E O QUE SE MEDE")
print("=" * 70)
for q, row in diz.iterrows():
    print(f"  diz que lembra {ESCALA[int(q)]:<24} ({int(row['pessoas']):>2} pessoas) · lembrou em {vir(row['lembrou'] * 100, 0)}% das manhãs gravadas")
print(f"  Quem disse que lembra 'uma vez por mês' ou menos ({baixo.sum()} pessoas) lembrou em {vir(p.loc[baixo, 'k'].sum() / p.loc[baixo, 'n'].sum() * 100, 0)}% das manhãs")
print(f"  Correlação de Spearman entre o que diz e o que mediu: {vir(rho_s.statistic, 2)}")

# CAMADA 3 · Quem lembra mais
def por_tercil(col):
    q = pd.qcut(p[col], 3, labels=['baixo', 'médio', 'alto'])
    return {k: (g['k'].sum() / g['n'].sum()) for k, g in p.groupby(q, observed=True)}
terc_atitude, terc_divaga = por_tercil('atitude'), por_tercil('divaga')
estacao_mes = {12: 'inverno', 1: 'inverno', 2: 'inverno', 3: 'primavera', 4: 'primavera', 5: 'primavera',
               6: 'verão', 7: 'verão', 8: 'verão', 9: 'outono', 10: 'outono', 11: 'outono'}
m['estacao'] = m['data'].dt.month.map(estacao_mes)
estacoes = m.groupby('estacao')['lembrou'].agg(['mean', 'size']).reindex(['inverno', 'primavera', 'verão', 'outono'])
p['faixa'] = pd.cut(p['idade'], [17, 25, 35, 45, 70], labels=['18 a 25', '26 a 35', '36 a 45', '46 a 69'])
idade = p.groupby('faixa', observed=True).apply(lambda g: pd.Series({'pessoas': len(g), 'lembrou': g['k'].sum() / g['n'].sum(), 'branco': 1 - g['c'].sum() / g['k'].sum()}))
print("\n" + "=" * 70)
print("  CAMADA 3 · QUEM LEMBRA MAIS")
print("=" * 70)
print("  Atitude em relação aos sonhos (tercis): " + " · ".join(f"{k} {vir(v * 100, 0)}%" for k, v in terc_atitude.items()))
print("  Mente que divaga (tercis):              " + " · ".join(f"{k} {vir(v * 100, 0)}%" for k, v in terc_divaga.items()))
print("  Estação do ano (Itália, hemisfério norte): " + " · ".join(f"{e} {vir(r['mean'] * 100, 0)}%" for e, r in estacoes.iterrows()))
for f_, r in idade.iterrows():
    print(f"  {f_:<8} ({int(r['pessoas']):>3} pessoas) · lembrou {vir(r['lembrou'] * 100, 0)}% · sonho em branco entre as lembranças {vir(r['branco'] * 100, 0)}%")

# CAMADA 4 · Outra amostra
print("\n" + "=" * 70)
print("  CAMADA 4 · OUTRA AMOSTRA E O LABORATÓRIO")
print("=" * 70)
print(f"  Nielsen 2012 ({mil(NIELSEN['n'])} pessoas): a lembrança sobe até os 20 a 29 anos e cai nas duas décadas seguintes.")
print(f"  No laboratório, acordando no REM: sonho com conteúdo em cerca de {vir(LAB['rem'] * 100, 0)}% das vezes; fora do REM, {vir(LAB['nrem'] * 100, 0)}%.")

In [ ]:
# --- VISUALIZAÇÃO ---

# GRÁFICO 1 · Cada pessoa tem a sua chance
fig1, ax1 = plt.subplots(figsize=(12, 8))
ax1.hist(p['taxa'] * 100, bins=np.linspace(0, 100, 16), density=True, color=DOURADO, alpha=0.75, edgecolor='white', label='taxa observada em 15 manhãs')
xs = np.linspace(0.001, 0.999, 400)
ax1.plot(xs * 100, pessoa.pdf(xs) / 100, color=VERDE, lw=3, label=f'beta-binomial, média {vir(media_bb * 100)}%')
ax1.axvline(media_bb * 100, color=VERDE, ls='--', lw=1.5)
ax1.set_xlabel('Manhãs em que a pessoa lembrou de ter sonhado (%)', fontsize=12)
ax1.set_ylabel('Densidade', fontsize=12)
ax1.set_title('Sete em cada dez manhãs começam com um sonho na lembrança\n204 pessoas, 2.900 manhãs gravadas', fontsize=14, pad=16)
ax1.legend(frameon=False, fontsize=11, loc='upper left')
plt.figtext(0.5, 0.01, RODAPE, ha='center', fontsize=9, color='gray')
plt.tight_layout(rect=(0, 0.03, 1, 1))
plt.savefig('dia-148-grafico-01-a-manha.png', dpi=150, bbox_inches='tight'); plt.close()
print("✅ Gráfico 1 salvo!")

# GRÁFICO 2 · O que se diz e o que se mede
fig2, ax2 = plt.subplots(figsize=(12, 8))
xq = diz.index.to_numpy().astype(int)
ax2.bar(xq, diz['lembrou'] * 100, color=[CINZA if q <= 2 else (DOURADO if q <= 4 else VERDE) for q in xq], width=0.65)
for q, row in diz.iterrows():
    ax2.text(q, row['lembrou'] * 100 + 1.5, f"{vir(row['lembrou'] * 100, 0)}%", ha='center', fontsize=13, fontweight='bold')
    ax2.text(q, 4, f"{int(row['pessoas'])}\npessoas", ha='center', fontsize=10, color='white')
ax2.set_xticks(xq); ax2.set_xticklabels([ESCALA[q].replace(' por ', '\npor ').replace('menos de ', 'menos de\n') for q in xq], fontsize=10)
ax2.set_ylim(0, 100); ax2.set_xlabel('O que a pessoa disse no questionário', fontsize=12)
ax2.set_ylabel('Manhãs em que lembrou de ter sonhado no diário (%)', fontsize=12)
ax2.set_title('Quem diz que quase nunca lembra, lembra em mais da metade das manhãs\nquando grava o que lembra toda manhã', fontsize=14, pad=16)
plt.figtext(0.5, 0.01, RODAPE, ha='center', fontsize=9, color='gray')
plt.tight_layout(rect=(0, 0.03, 1, 1))
plt.savefig('dia-148-grafico-02-o-que-se-diz.png', dpi=150, bbox_inches='tight'); plt.close()
print("✅ Gráfico 2 salvo!")

# GRÁFICO 3 · Quem lembra mais
fig3, axs = plt.subplots(1, 3, figsize=(14, 7), sharey=True)
for ax, (tit, dic, cor) in zip(axs, [('Atitude em relação\naos sonhos', terc_atitude, VERDE),
                                      ('Mente que divaga', terc_divaga, DOURADO),
                                      ('Estação do ano\n(hemisfério norte)', estacoes['mean'].to_dict(), CINZA)]):
    ks, vs = list(dic.keys()), [v * 100 for v in dic.values()]
    ax.bar(range(len(ks)), vs, color=cor, width=0.6)
    for i, v in enumerate(vs): ax.text(i, v + 1.2, f"{vir(v, 0)}%", ha='center', fontsize=12, fontweight='bold')
    ax.set_xticks(range(len(ks))); ax.set_xticklabels(ks, fontsize=11); ax.set_title(tit, fontsize=13)
    ax.set_ylim(0, 95)
axs[0].set_ylabel('Manhãs em que lembrou de ter sonhado (%)', fontsize=12)
fig3.suptitle('Quem dá importância aos sonhos e quem divaga lembra mais', fontsize=15, y=0.99)
plt.figtext(0.5, 0.01, RODAPE, ha='center', fontsize=9, color='gray')
plt.tight_layout(rect=(0, 0.03, 1, 0.95))
plt.savefig('dia-148-grafico-03-quem-lembra-mais.png', dpi=150, bbox_inches='tight'); plt.close()
print("✅ Gráfico 3 salvo!")

# GRÁFICO 4 · Uma semana
fig4, ax4 = plt.subplots(figsize=(12, 8))
kk = np.arange(8); pk = np.array([prob_k_de_n(k, 7) for k in kk]) * 100
ax4.bar(kk, pk, color=[VERDE if k >= 5 else DOURADO if k >= 3 else VERMELHO for k in kk], width=0.65)
for k, v in zip(kk, pk): ax4.text(k, v + 0.6, f"{vir(v)}%", ha='center', fontsize=12, fontweight='bold')
ax4.set_xticks(kk); ax4.set_xlabel('Manhãs da semana em que você lembra de ter sonhado', fontsize=12)
ax4.set_ylabel('Probabilidade (%)', fontsize=12)
ax4.set_title('Numa semana, o mais provável é lembrar em cinco manhãs ou mais\npassar a semana inteira sem lembrar é raro', fontsize=14, pad=16)
plt.figtext(0.5, 0.01, 'Fonte: modelo beta-binomial sobre Elce et al. (2025) · cálculo próprio | #365Probabilidades', ha='center', fontsize=9, color='gray')
plt.tight_layout(rect=(0, 0.03, 1, 1))
plt.savefig('dia-148-grafico-04-uma-semana.png', dpi=150, bbox_inches='tight'); plt.close()
print("✅ Gráfico 4 salvo!")

### 💡 O Insight

**Numa manhã qualquer, a chance de você acordar lembrando que sonhou é de 72% (IC 95% de
69% a 74%). Em 58% das manhãs o sonho vem com conteúdo; em 14%, só a certeza de ter sonhado.
Numa semana, a chance de não lembrar de nenhum sonho é de menos de 1%.**

A chance não é igual para todo mundo. Uma em cada seis pessoas lembra em mais de nove de
cada dez manhãs. Lembra mais quem dá importância aos sonhos (78% no terço mais interessado
contra 65% no menos interessado) e quem tem a mente que divaga (80% contra 68%). No inverno
italiano, a lembrança caiu para 64%.

E tem o que as pessoas dizem de si. Quem afirmou no questionário que lembra dos sonhos uma
vez por mês ou menos lembrou em 60% das manhãs gravadas. Parte do esquecimento parece estar
no hábito de não reparar, não na falta de sonho.

*Se amanhã você gravasse o que lembra antes de pegar o celular, o que encontraria?*

---

### ⚠️ Limitações do Modelo

- A amostra é italiana, de voluntários, mais jovem que a população (idade média 35) e
  recrutada boca a boca. Pode haver mais interesse por sonhos do que no geral.
- O próprio diário muda o resultado. Quem sabe que vai gravar o sonho toda manhã presta
  mais atenção, e por isso lembra mais. A camada 2 mostra o tamanho disso, mas não separa
  o efeito de prestar atenção do erro de memória do questionário.
- A coleta começou em março de 2020, no início da pandemia, quando a lembrança de sonhos
  subiu no mundo inteiro (os autores citam um estudo com 19.355 adultos).
- A estação foi definida pela data de cada manhã, na Itália. No Brasil as estações são
  invertidas, e o efeito pode vir da luz, não do calendário.
- As comparações por tercil e por faixa de idade são descritivas, sem os controles do
  modelo dos autores. No modelo deles, a idade prevê lembrar o conteúdo, não lembrar de
  ter sonhado.
- Os números de Nielsen 2012 foram lidos no resumo; o artigo não foi lido inteiro.
- Os 85% e 45% do laboratório vêm de Siclari et al. 2013, citados por Elce et al.

*A ciência é honesta sobre o que não sabe. O modelo também.*

---

### 📎 Links
- Substack: [link do post]
- Instagram: [link do post]

---
*365 Probabilidades · Decidindo com dados, um dia de cada vez.*